### This notebook demonstrates how to build and evaluate a neural network regression model using TensorFlow and Keras. We will follow these steps:
1. Load and preprocess the data
2. Build the neural network model
3. Train the model
4. Evaluate the model's performance
5. Perform cross-validation and hyperparameter tuning

### 1. Load and prepare the dataset ###

In [29]:
### Import libraries ###
import pandas as pd
import numpy as np
import helper_func as hf
import os
import time
import random

from scipy.stats import kendalltau

from sklearn.metrics import mean_squared_error, r2_score


import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from torch import nn, optim
import torch.nn.functional as F
from torchinfo import summary

In [31]:
### define data path and loader ###
data_path = './processed_data/dataset_with_descriptors.csv'

# Path to save the model checkpoints and parameters
nn_checkpoint_path = './nn_output/checkpoints'
os.makedirs(nn_checkpoint_path, exist_ok=True)

model_path = './nn_output/nn_model_params'
os.makedirs(model_path, exist_ok=True)

In [ ]:
class EarlyStopping():
    """Early stopping utility to stop training when validation loss does not improve."""
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
def get_data_loader(data_path: str,
                    batch_size: int = 32,
                    test_size: float = 0.1,
                    val_size: float = 0.1,
                    random_state=42):
    """
    Load and preprocess the dataset, then create DataLoaders for training, validation, and testing.
    :param data_path:
    :param batch_size:
    :param test_size:
    :param val_size:
    :param random_state:
    :return: data loaders for training, validation, and testing
    """
    # 1. Read the data and preprocess
    assert data_path.endswith('.csv') and os.path.exists(data_path), "Data path must be a valid CSV file."
    data = pd.read_csv(data_path, low_memory=False)
    features = [col for col in data.columns if col != 'Mole fraction']
    target = 'Mole fraction'
    data = data.dropna(subset=features)
    X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
    X = hf.convert_to_numeric(X)
    y = data[target]
    X_np = X.to_numpy(dtype=float)
    # 2. Convert to PyTorch tensors
    X_tensor = torch.tensor(X_np, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

    # 3. Create a TensorDataset and split into train/val/test
    dataset = TensorDataset(X_tensor, y_tensor)
    total_size = len(dataset)
    test_size = int(test_size * total_size)
    val_size = int(val_size * total_size)
    train_size = total_size - test_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(random_state))

    # 4. Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader

def eval_nn(data_loader,
            model,
            device,
            print_out: bool = False) -> tuple:
    """Evaluate a neural network model on a given dataset.
     :param data_loader: DataLoader providing the evaluation dataset.
     :param model: The neural network model to evaluate.
     :param device: The device (CPU or GPU) to perform evaluation on.
     :return: MSE and R2 (prints the Mean Squared Error and R^2 Score)."""

    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    mse = mean_squared_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    if print_out:
        print(f"Mean Squared Error: {mse:.4f}")
        print(f"R^2 Score: {r2:.4f}")

    return (mse, r2)


def train_nn(model,
             train_loader,
             val_loader,
             device,
             num_epochs: int = 50,
             learning_rate: float = 0.001,
             opt_cls = optim.Adam,
             early_stopping_patience: int = 5,
             checkpoint_path=nn_checkpoint_path):
    """Train a neural network model using the provided training and validation data loaders.
     :param model: The neural network model to train.
     :param train_loader: DataLoader providing the training dataset.
     :param val_loader: DataLoader providing the validation dataset.
     :param device: The device (CPU or GPU) to perform training on.
     :param num_epochs: Number of epochs to train the model.
     :param learning_rate: Learning rate for the optimizer.
     :param checkpoint_path: Path to save model checkpoints during training."""

    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validate the model
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}, Validation Loss: {val_loss:.4f}")

        # Save the best model checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(checkpoint_path, 'best_model.pth'))

In [26]:
class BaselineNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], output_dim=1):
        super(BaselineNN, self).__init__()
        self.name = "SimpleMLP"
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.fc1 = nn.Linear(self.input_dim, self.hidden_dims[0])
        self.fc2 = nn.Linear(self.hidden_dims[0], self.hidden_dims[1])
        self.fc3 = nn.Linear(self.hidden_dims[1], self.output_dim)


    def forward(self, x):
        x = x.view(-1, self.input_dim)  # Flatten the input

        x = F.relu(self.fc1(x)) # Layer 1 with ReLU activation
        x = F.relu(self.fc2(x)) # Layer 2 with ReLU activation
        return self.fc3(x) # Output layer without activation (for regression)

In [27]:
train_loader, val_loader, test_loader = get_data_loader(data_path, batch_size=32)
input_dim = next(iter(train_loader))[0].shape[1]
model = BaselineNN(input_dim=input_dim, hidden_dims=[64, 32], output_dim=1)
print(summary(model, input_size=(1, input_dim)))

Layer (type:depth-idx)                   Output Shape              Param #
BaselineNN                               [1, 1]                    --
├─Linear: 1-1                            [1, 64]                   27,520
├─Linear: 1-2                            [1, 32]                   2,080
├─Linear: 1-3                            [1, 1]                    33
Total params: 29,633
Trainable params: 29,633
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.03
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.12
Estimated Total Size (MB): 0.12
